<a href="https://colab.research.google.com/github/chetools/CHE4071_Spring2026/blob/main/PPO_2Tanks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
!pip install -U flax

In [34]:
from flax import nnx
import jax
from jax.flatten_util import ravel_pytree
import jax.numpy as jnp
import jax.random as jr
import numpy as np
import einops
import optax
from optax import contrib
from plotly.subplots import make_subplots
import plotly.io as pio
import orbax.checkpoint as ocp
pio.templates.default = "plotly_dark"
eps32=jnp.finfo(jnp.float32).eps
np.set_printoptions(precision=3,suppress=True, linewidth=200)

In [35]:
class LayerWithDropout(nnx.Module):
    def __init__(self, din, dout, rngs, dropout=0.1):
        self.layer = nnx.Linear(din,dout,rngs=rngs)
        self.dropout = nnx.Dropout(dropout, rngs=rngs)

    def __call__(self, x):
        return self.dropout(self.layer(x))

symlog = lambda x: jnp.sign(x)*jnp.log(1+jnp.abs(x))
symexp = lambda x: jnp.sign(x)*(jnp.exp(jnp.abs(x))-1)
pack = lambda *x: jnp.concat([jax.flatten_util.ravel_pytree(a)[0] for a in x])
linspace_vec = jnp.vectorize(jnp.linspace, signature='(),(),()->(n)')


def calc_prob(x, B, probs):
    return probs[jnp.searchsorted(B,x,side='left')]
calc_prob=jnp.vectorize(calc_prob, signature='(),(nbins),(nbins)->()')

def twohot(x,B):
    nbins=B.size
    idx = jnp.searchsorted(B,x)
    idx_lo = jnp.clip(idx-1,0,nbins-1)
    idx_hi = jnp.clip(idx,0,nbins-1)
    x_lo = (x-B[idx_lo])
    w_hi = jnp.where(x_lo<eps32, 1., x_lo/(B[idx_hi]-B[idx_lo]+eps32))
    w_lo = 1-w_hi
    return jnp.zeros_like(B).at[idx_lo].set(w_lo).at[idx_hi].set(w_hi)

twohot = jnp.vectorize(twohot, signature=f'(),(n)->(n)')

interp = jnp.vectorize(jnp.interp, signature='(),(nbins),(nbins)->()')
interp2 = jnp.vectorize(interp, signature='(dout),(dout,nbins),(dout,nbins)->(dout)')

def sample(probs, B, key, Nsamples=1):
    #probs: (Nbatch, dout, nbins)
    #B: (dout, nbins)
    Nbatch = probs.shape[0]
    dout = probs.shape[1]
    cumprobs = jnp.cumsum(probs, axis=-1)
    r = jr.uniform(key, (Nsamples, Nbatch, dout))
    return jnp.squeeze(interp2(r, cumprobs, B))
class CategoricalValue(nnx.Module):
    def __init__(self, din, dout,rngs, range=None, nbins=100, symlogscale=False, equal=True, init=None):
        self.nbins=nbins
        self.dout=dout
        self.rngs=rngs
        if range is None:
            range = np.tile(((-20,20)),(dout,1))
        if symlogscale:
            self.B = symexp(linspace_vec(symlog(range[:, 0]), symlog(range[:, 1]), self.nbins))
        else:
            self.B = linspace_vec(range[:, 0], range[:, 1], self.nbins)
        self.twohot=lambda x: twohot(x,self.B)
        self.layer = nnx.LinearGeneral(din,(dout,self.nbins), rngs=rngs)
        if equal:
            kernel=self.layer.kernel.get_value()
            self.layer.kernel.set_value(jax.lax.broadcast(jnp.mean(kernel),kernel.shape))
        if init is not None:
            self.layer.kernel.set_value(self.layer.kernel.get_value()*1e-6)
            twohot_init = self.twohot(init)
            self.layer.bias.set_value(jnp.where(twohot_init>eps32, twohot_init, -10))


    def probs(self, x):
        return nnx.softmax(self.layer(x),axis=-1)

    def sample(self, x, key, Nsamples=1):
        return sample(self.probs(x), self.B, key, Nsamples)

    def value_probs(self, x):
        probs = self.probs(x)
        return jnp.einsum('kj,...kj->...k', self.B, probs), probs

    def __call__(self,x):
        return jnp.einsum('kj,...kj->...k', self.B, self.probs(x))

class CategoricalValueMLP(nnx.Module):
    def __init__(self, dims, rngs, signature=None, dropout=0.1, range=None, nbins=100, symlogscale=False, equal=True, init=None):
        self.pack = jnp.vectorize(pack, signature=signature)
        self.nbins=nbins
        self.rngs=rngs
        self.layers = nnx.List([nnx.Linear(din, dout, rngs=rngs) for din, dout in zip(dims[:-2],dims[1:-1])])
        self.cv_layer = CategoricalValue(dims[-2],dims[-1], rngs=rngs, range=range, nbins=nbins, symlogscale=symlogscale, equal=equal, init=init)


    def eval_layers(self, *x):
        x= self.pack(*x)
        for layer in self.layers:
            x=nnx.silu(layer(x))
        return x

    def probs(self, *x):
        return self.cv_layer.probs(self.eval_layers(*x))

    def sample(self, *x, **kwargs):
        Nsamples=kwargs.get('Nsamples',1)
        key = kwargs.get('key', jr.PRNGKey(0))
        return sample(self.probs(*x), self.cv_layer.B, key, Nsamples)

    def value_probs(self, *x):
        return self.cv_layer.value_probs(self.eval_layers(*x))

    def __call__(self,*x):
        return self.cv_layer(self.eval_layers(*x))

In [36]:
def reward(t, y, u, ysp, dt):
    err =y[1]/ysp-1.
    setpoint_error = jnp.abs(err)
    return -(0.1*u[1]/qmax[1]+setpoint_error)*dt
reward=jax.jit(jnp.vectorize(reward, signature='(),(2),(2),(),()->()'))

alpha = 0.3
A = 2.

@jax.jit
def f(t, y, u):
    y1, y2 = y
    qin1, qin2 = u
    d1 = y1-y2
    q1 = alpha*jnp.copysign(jnp.sqrt(jnp.abs(d1)),d1)
    q2 = jnp.where(y2<0., 0., alpha*jnp.sqrt(y2))
    return jnp.array([(qin1-q1), (qin2+q1-q2)])/A

Nsteps=50
tend=50.
dt=tend/Nsteps
key=jax.random.PRNGKey(1)
qmax = jnp.array([4.,1.])
ymax = 100
rngs = nnx.Rngs(0)

def step(y, u, dt):
    return y+f(0, y, u)*dt
step=jax.jit(jax.vmap(step, (0,0,None)))

In [37]:
Nbuf=2

@jax.jit
def Atrain2(Astate, start, ysp, e):
    ANN, Aopt = nnx.merge(Agraph, Astate)

    def scan_f(carry, key):
            nt, ybuf0, ysp = carry
            t=jnp.full_like(ysp, nt*dt)
            u0 = ANN.sample(ybuf0/ymax, ysp/ymax, key=key)
            y1 = step(ybuf0[:,0],u0,dt)
            ybuf1=jnp.roll(ybuf0,1, axis=0)
            ybuf1=ybuf1.at[:,0].set(y1)
            return (nt+1, ybuf1, ysp), (t, ybuf0, u0, ybuf1, ysp)

    _, (t, ybuf0,u0, ybuf1, ysp)= jax.lax.scan(scan_f, (0, jnp.repeat(start[:,None,:],Nbuf,axis=1), ysp), jax.random.split(key,Nsteps)) #Nstep, Nstart, 2
    r = reward(t, ybuf1[:,:,0], u0, ysp, dt) #Nstep, Nstart,
    du1 = jnp.diff(u0, n=1, axis=0, prepend=jnp.tile(u0[0], (1,1,1)))
    # du1 = jnp.where(jnp.abs(du1)<0.2*qmax, 0, du1)
    du1diff = jnp.sum(jnp.cumsum(du1**2, axis=0)-(jnp.cumsum(du1, axis=0))**2,axis=-1)

    r2go = jnp.cumsum(r[::-1],axis=0)[::-1] - 0.001*du1diff
    mean_r2go = jnp.mean(r2go,axis=1)
    adv = (r2go - mean_r2go[:,None])
    oldsum_logprob=jnp.sum(jnp.log(calc_prob(u0, ANN.cv_layer.B, ANN.probs(ybuf0/ymax, ysp/ymax))),axis=-1)

    def L(ANN):
        sum_logprob=jnp.sum(jnp.log(calc_prob(u0, ANN.cv_layer.B, ANN.probs(ybuf0/ymax, ysp/ymax))),axis=-1)
        s=jnp.exp(sum_logprob-oldsum_logprob)
        res = -jnp.sum(jnp.minimum(s*adv, jnp.clip(s, 1-e, 1+e)*adv))
        return res

    val = L(ANN)
    grads=nnx.grad(L)(ANN)

    Aopt.update(ANN, grads)
    Astate = nnx.state((ANN, Aopt))
    return val, Astate, jnp.mean(mean_r2go)

In [44]:

checkpointer = ocp.StandardCheckpointer()
ANN=CategoricalValueMLP((5,50,20,2), signature='(2,2),()->(5)', range=np.array([(0.,qmax[0]), (0,qmax[1])]), equal=False, symlogscale=False,
                        rngs=nnx.Rngs(22), nbins=51)
Aopt = nnx.Optimizer(ANN, optax.adam(1e-3),wrt=nnx.Param)
Agraph, Astate = nnx.split((ANN, Aopt))
key=jr.PRNGKey(0)
best = -jnp.inf

In [45]:
lr=1e-3
opt = optax.inject_hyperparams(optax.adam)(learning_rate=lr)
Aopt = nnx.Optimizer(ANN, opt, wrt=nnx.Param)
ckpt_dir = ocp.test_utils.erase_and_create_empty('/tmp/checkpoints')
Agraph, Astate = nnx.split((ANN, Aopt))
checkpointer.save(ckpt_dir/f'PPO_2Tanks', Astate)
grid = jnp.mgrid[1:15:7j, 1:9:4j, 3:8:6j].swapaxes(0,-1).reshape(-1,3)
lr=1e-3
e=0.1
Nrepeat=100


In [55]:
Ngrid = grid.shape[0]
Agraph, Astate = nnx.split((ANN, Aopt))
for i in range(3000):
    sum_meanr2go=0.
    idx=np.argsort(np.random.uniform(0,1,(Ngrid)))
    for grid_item in grid[idx]:
        start=grid_item[:2]
        ysp=grid_item[2]
        val, Astate, mean_r2go = Atrain2(Astate, jnp.tile(start, (Nrepeat,1)), jnp.repeat(ysp,Nrepeat), e)
        sum_meanr2go += mean_r2go
    grid_meanr2go = sum_meanr2go/Ngrid
    print(i, grid_meanr2go, best, lr)
    if i % 1 ==0:
        if grid_meanr2go>=best:
            best=grid_meanr2go
            ckpt_dir = ocp.test_utils.erase_and_create_empty('/tmp/checkpoints')
            checkpointer.save(ckpt_dir/f'PPO_2Tanks', Astate)
        else:
            pass
            lr/=2
            e/=2
            Astate = checkpointer.restore(ckpt_dir/f'PPO_2Tanks', Astate)
            Astate[1]['opt_state']['hyperparams']['learning_rate']=lr

0 -2.0591688 -7.5090666 0.001
1 -2.058946 -2.0591688 0.001
2 -2.0591378 -2.058946 0.001
3 -2.0719101 -2.058946 0.0005


KeyboardInterrupt: 

In [54]:
Astate = checkpointer.restore(ckpt_dir/f'PPO_2Tanks', Astate)
ANN, Aopt=nnx.merge(Agraph,Astate)

In [56]:
def scan_f(carry, key):
        nt, ybuf0, ysp = carry
        t=jnp.full_like(ysp, nt*dt)
        # u0 = ANN.sample(ybuf0/ymax, ysp/ymax, key=key).reshape(1,-1)
        u0 = ANN(ybuf0/ymax, ysp/ymax)
        y1 = step(ybuf0[:,0],u0,dt)
        ybuf1=jnp.roll(ybuf0,1, axis=0)
        ybuf1=ybuf1.at[:,0].set(y1)
        return (nt+1, ybuf1, ysp), (t, ybuf0, u0, ybuf1, ysp)

start=jnp.array([[7.,2.]])
ysp=7.

_, (t, ybuf0,u0, ybuf1, ysp)= jax.lax.scan(scan_f, (0, jnp.repeat(start[:,None,:],Nbuf,axis=1), ysp), jax.random.split(key,Nsteps)) #Nstep, Nstart, 2


In [57]:

Ngrid = 0
fig = make_subplots(rows=1,cols=2)
fig.add_scatter(x=np.linspace(0,tend,Nsteps), y=ybuf0[:,Ngrid,0,0],row=1,col=1,name='h1', mode='markers+lines')
fig.add_scatter(x=np.linspace(0,tend,Nsteps), y=ybuf0[:,Ngrid,0,1],row=1,col=1,name='h2', mode='markers+lines')
fig.add_scatter(x=np.linspace(0,tend,Nsteps), y=u0[:,Ngrid,0],row=1,col=2,name='q1', mode='markers+lines')
fig.add_scatter(x=np.linspace(0,tend,Nsteps), y=u0[:,Ngrid,1],row=1,col=2,name='q2', mode='markers+lines')
fig.show()